In [2]:
import folium
import pandas as pd

map = folium.Map(location=[37, 127], zoom_start=7)
marker = folium.Marker([37.341435483, 126.733026596],
                       popup='한국공학대학교',
                       icon = folium.Icon(color='blue'))

marker.add_to(map)
map.save(r'/content/drive/MyDrive/python/Folium Map/학교 디지털 콘텐츠 공모전/2022_빅데이터/uni_map.html')

map

In [3]:
from folium import Circle

In [4]:
import pandas as pd
import folium

filePath =r'/content/drive/MyDrive/python/Folium Map/학교 디지털 콘텐츠 공모전/2022_빅데이터/대학교주소좌표.xlsx'
df_from_excel = pd.read_excel(filePath, engine='openpyxl', header=None)

df_from_excel.columns = ['학교이름', '주소', 'x', 'y']

name_list = df_from_excel['학교이름'].to_list()
addr_list = df_from_excel['주소'].to_list()
position_x_list = df_from_excel['x'].to_list()
position_y_list = df_from_excel['y'].to_list()

map = folium.Map(location=[37, 127], zoom_start=7)

for i in range(len(name_list)):
    circle = Circle(location = [position_y_list[i], position_x_list[i]],
                    radius = 50,
                    popup = name_list[i],
                    color = 'blue')
#                    color = color_select(i))
    circle.add_to(map)

map


In [5]:
map2 = r'/content/drive/MyDrive/python/Folium Map/학교 디지털 콘텐츠 공모전/2022_빅데이터/uni_map.html'

In [6]:
map

In [9]:
import pandas as pd
import numpy as np
import platform
import matplotlib.pyplot as plt

import folium

population = pd.read_excel('/content/drive/MyDrive/python/Folium Map/학교 디지털 콘텐츠 공모전/2022_빅데이터/05. population_raw_data.xlsx', header=1)
population.fillna(method='pad', inplace=True)

population.rename(columns ={'행정구역(동읍면)별(1)':'광역시도',
                            '행정구역(동읍면)별(2)':'시도',
                            '계':'인구수'}, inplace=True)

population = population[(population['시도'] != '소계')]

In [10]:
from os import popen

population.is_copy = False

population.rename(columns = {'항목':'구분'}, inplace = True)

population.loc[population['구분'] == '총인구수 (명)', '구분'] = '합계'
population.loc[population['구분'] == '남자인구수 (명)', '구분'] = '남자'
population.loc[population['구분'] == '여자인구수 (명)', '구분'] = '여자'

In [11]:
population['20-39세'] = population['20 - 24세'] + population['25 - 29세'] + population['30 - 34세'] + population['35 - 39세']

population['65세이상'] = population['65 - 69세'] + population['70 - 74세'] + population['75 - 79세'] + \
population['80 - 84세'] + population['85 - 89세'] + population['90 - 94세'] + population['95 - 99세'] + population['100+']

In [ ]:
pop = pd.pivot_table(population,
                     index = ['광역시도', '시도'],
                     columns = ['구분'],
                     values = ['인구수', '20-39세', '65세이상'])

In [ ]:
pop['소멸비율'] = pop['20-39세', '여자'] / (pop['65세이상', '합계'] / 2)

In [ ]:
pop['소멸위기지역'] = pop['소멸비율'] < 1.0

In [ ]:
pop[pop['소멸위기지역']==True].index.get_level_values(1)

In [ ]:
pop.reset_index(inplace=True)

In [ ]:
tmp_columns = [pop.columns.get_level_values(0)[n] +
               pop.columns.get_level_values(1)[n]
               for n in range(0, len(pop.columns.get_level_values(0)))]

pop.columns = tmp_columns

In [ ]:
si_name = [None] * len(pop)

tmp_gu_dict = {'수원':['장안구', '권선구', ' 팔달구', '영통구'],
               '성남':['수정구', '중원구', '분당구'],
               '안양':['만안구', '동안구'],
               '안산':['상록구', '단원구'],
               '고양':['덕양구', '일산동구', '일산서구'],
               '용인':['처인구', '기흥구', '수지구'],
               '청주':['상당구', '서원구', '흥덕구', '청원구'],
               '천안':['동남구', '서북구'],
               '전주':['완산구', '덕진구'],
               '포항':['남구', '북구'],
               '창원':['의창구', '성산구', '마산합포구', '마산회원구'],
               '부천':['오정구', '원미구', '소사구']}

In [ ]:
for n in pop.index:
    if pop['광역시도'][n][-3:] not in ['광역시', '특별시', '자치시']:
        if pop['시도'][n][:-1] == '고성' and pop['광역시도'][n] == '강원도':
            si_name[n] = '고성(강원)'
        elif pop['시도'][n][:-1] == '고성' and pop['광역시도'][n] == '경상남도':
            si_name[n] = '고성(경남)'
        else:
            si_name[n] = pop['시도'][n][:-1]

        for keys, values in tmp_gu_dict.items() :
            if pop['시도'][n] in values:
                if len(pop['시도'][n]) == 2:
                    si_name[n] = keys + ' ' + pop['시도'][n]
                elif pop['시도'][n] in ['마산합포구', '마산회원구']:
                    si_name[n] = keys + ' ' + pop['시도'][n][2:-1]
                else:
                    si_name[n] = keys + ' ' + pop['시도'][n][:-1]

    elif pop['광역시도'][n] == '세종특별자치시':
        si_name[n] = '세종'

    else:
        if len(pop['시도'][n]) == 2:
            si_name[n] = pop['광역시도'][n][:2] + ' ' + pop['시도'][n]
        else:
            si_name[n] = pop['광역시도'][n][:2] + ' ' + pop['시도'][n][:-1]

In [ ]:
pop['ID'] = si_name

In [ ]:
pop_folium = pop.set_index('ID')

In [ ]:
import json

In [ ]:
geo_path = '/content/drive/MyDrive/python/Folium Map/05. skorea_municipalities_geo_simple.json'
geo_str = json.load(open(geo_path, encoding='utf-8'))

F_map = folium.Map(location=[36.2002, 127.054], zoom_start=7)
F_map.choropleth(geo_data = geo_str,
               data = pop_folium['인구수합계'],
               columns = [pop_folium.index, pop_folium['인구수합계']],
               fill_color = 'YlGnBu', #PuRd, YlGnBu
               key_on = 'feature.id')

F_map

In [ ]:
map2 = folium.Map(location=[36.2002, 127.054], size=[400, 300], zoom_start=7)
map2.choropleth(geo_data = geo_str,
               data = pop_folium['소멸위기지역'],
               columns = [pop_folium.index, pop_folium['소멸위기지역']],
               fill_color = 'YlGnBu', #PuRd, YlGnBu
               key_on = 'feature.id')

map2

In [ ]:
for i in range(len(name_list)):
    circle = Circle(location = [position_y_list[i], position_x_list[i]],
                    radius = 50,
                    popup = name_list[i],
                    color = 'red')
    circle.add_to(map2)

map2